# Sprint 4 — Arquitetura GPT

Projeto Integrador — Construção de um Large Language Model (LLM) From Scratch

Baseado no Capítulo 4 (*Implementing a GPT model to generate text*) do livro **Build a Large Language Model (From Scratch)**, de Sebastian Raschka.

Este notebook reúne, em sequência, os códigos dos Entregáveis 2 a 4 da sprint (`src/Sprint 4/4,1.py` a `4,3.py`): Layer Normalization, FeedForward com GELU, conexões residuais, o Transformer Block completo, o modelo GPT (`GPTModel`) e a geração de texto autorregressiva. O Entregável 5 (experimentação) está em `src/Sprint 4/4,5_experimentos.py`, com os resultados documentados em `relatório/Sprint 4/`.

## 4.2 — Layer Normalization

In [2]:
import torch
import torch.nn as nn


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

### Aplicando — média e variância antes/depois da normalização

In [3]:
torch.manual_seed(123)
batch_exemplo = torch.randn(2, 5)

ln = LayerNorm(emb_dim=5)
saida_ln = ln(batch_exemplo)

media = saida_ln.mean(dim=-1, keepdim=True)
variancia = saida_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Média após LayerNorm (deve ser ~0):", media)
print("Variância após LayerNorm (deve ser ~1):", variancia)

Média após LayerNorm (deve ser ~0): tensor([[-2.9802e-08],
        [ 0.0000e+00]], grad_fn=<MeanBackward1>)
Variância após LayerNorm (deve ser ~1): tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


## 4.3 — FeedForward Network com ativação GELU

In [4]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

## 4.4 — Conexões residuais (shortcut connections)

Em vez de uma classe própria, as conexões residuais aparecem direto no `forward()` do Transformer Block, como `x = x + shortcut` ao redor da atenção e da FeedForward (seção 4.5). A ideia é simples: somar a entrada de uma subcamada à sua saída cria um atalho para o gradiente, evitando que ele desapareça em redes profundas como a de 12 blocos empilhados do GPT.

## 4.5 — Transformer Block (Entregável 2)

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"], num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

### Aplicando — shape de entrada e saída do bloco

In [7]:
import tiktoken

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

tokenizer = tiktoken.get_encoding("gpt2")
texto_inicial = "Era uma vez"
ids_entrada = torch.tensor(tokenizer.encode(texto_inicial)).unsqueeze(0)

torch.manual_seed(123)
camada_embedding = torch.nn.Embedding(GPT_CONFIG_124M["vocab_size"], GPT_CONFIG_124M["emb_dim"])
camada_posicional = torch.nn.Embedding(GPT_CONFIG_124M["context_length"], GPT_CONFIG_124M["emb_dim"])

token_embeddings = camada_embedding(ids_entrada)
pos_embeddings = camada_posicional(torch.arange(ids_entrada.shape[1]))
entrada_bloco = token_embeddings + pos_embeddings

torch.manual_seed(123)
bloco = TransformerBlock(GPT_CONFIG_124M)
saida_bloco = bloco(entrada_bloco)

print("Shape de entrada:", entrada_bloco.shape)
print("Shape de saída:", saida_bloco.shape)

Shape de entrada: torch.Size([1, 6, 768])
Shape de saída: torch.Size([1, 6, 768])


## 4.6 — Modelo GPT completo (Entregável 3)

In [8]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


def contar_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    return total

### Aplicando — shape dos logits e parâmetros (com e sem weight tying)

In [9]:
texto_inicial = "Era uma vez"
ids_texto = torch.tensor(tokenizer.encode(texto_inicial))
lote = torch.stack([ids_texto, ids_texto])

torch.manual_seed(123)
modelo = GPTModel(GPT_CONFIG_124M)
logits = modelo(lote)

print("Shape do lote de entrada:", lote.shape)
print("Shape dos logits de saída:", logits.shape)

total_parametros = contar_parametros(modelo)
print("Total de parâmetros (soma ingênua):", total_parametros)

parametros_com_weight_tying = (
    total_parametros - sum(p.numel() for p in modelo.out_head.parameters())
)
print("Total de parâmetros (com weight tying, como no GPT-2 real):", parametros_com_weight_tying)

Shape do lote de entrada: torch.Size([2, 6])
Shape dos logits de saída: torch.Size([2, 6, 50257])
Total de parâmetros (soma ingênua): 163009536
Total de parâmetros (com weight tying, como no GPT-2 real): 124412160


## 4.7 — Geração de texto (Entregável 4)

In [10]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

### Aplicando — gerando a partir de "Era uma vez"

In [11]:
torch.manual_seed(123)
modelo = GPTModel(GPT_CONFIG_124M)
modelo.eval()

ids_saida = generate_text_simple(
    model=modelo,
    idx=ids_entrada,
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"],
)
texto_gerado = tokenizer.decode(ids_saida.squeeze(0).tolist())

print("Texto inicial:", texto_inicial)
print("IDs gerados:", ids_saida)
print("Texto gerado (modelo sem treino, saída esperada = ruído):", texto_gerado)

Texto inicial: Era uma vez
IDs gerados: tensor([[   36,   430,   334,  2611,  1569,    89, 20583, 45767,  9650, 22783,
         24459, 43231, 47062, 34657, 18631, 18992]])
Texto gerado (modelo sem treino, saída esperada = ruído): Era uma vez curriculumDistanceotion Province declines Normandy Compton analogous bore improper


## Entregável 5 — Experimentação e análise

Os experimentos de arquitetura (variando `n_layers`, `n_heads`, `drop_rate` e `context_length` em cima do `GPTModel` definido acima) estão em `src/Sprint 4/4,5_experimentos.py`. Os resultados, a tabela comparativa e a análise de cada experimento estão documentados em `relatório/Sprint 4/Sprint4_Experimentacao.pdf` e `relatório/Sprint 4/Sprint4_Análise.pdf`.